# Demo 0 · The silent failure

**Slide:** "One notebook, two environments" (slide 3).

This notebook uses whatever Colab has preinstalled and trains a small Keras model **without any seeds**.
Run it twice, or once here and once on your laptop: the code is identical, the accuracy is not.

Runtime: CPU is enough. Takes about one minute.

In [ ]:
REPO_URL = "https://github.com/keila-moral/reproducible-ml-demo"
import os
if not os.path.exists("/content/repo"):
    !git clone -q {REPO_URL} /content/repo
else:
    !git -C /content/repo pull -q   # already cloned in this session: get the latest version
%cd /content/repo
!git log --oneline -1

## 1 · Build the dataset

The 8×8 handwritten-digits set ships with scikit-learn, so nothing is downloaded.

In [ ]:
!PYTHONPATH=src python -m repro_demo.prepare --out data/digits.parquet

## 2 · Which environment are we in?

Note these versions: they are whatever this Colab image happens to contain today.

In [ ]:
import os, sys, platform
os.environ["KERAS_BACKEND"] = "jax"
sys.path.insert(0, "src")

import jax, keras, numpy, sklearn
print("python ", platform.python_version())
print("jax    ", jax.__version__, jax.devices())
print("keras  ", keras.__version__, keras.backend.backend())
print("numpy  ", numpy.__version__)
print("sklearn", sklearn.__version__)

## 3 · A helper: train once in a fresh process

Each call starts a brand-new Python process, like a rerun on another day: nothing carries over between runs. Run this cell, no need to show it.

In [ ]:
import subprocess

def train_fresh(seed):
    """Train once in a new Python process and return the validation accuracy."""
    script = ("from repro_demo.train import TrainConfig, train; "
              f"print(train(TrainConfig(seed={seed}))['val_accuracy'])")
    out = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True,
                         env={**os.environ, "PYTHONPATH": "src"})
    if out.returncode != 0:
        raise RuntimeError(out.stderr[-2000:])
    return float(out.stdout.strip().splitlines()[-1])

## 4 · Train three times, no seed

`seed=None` means: random train/validation split, random weight initialisation, random dropout.

In [ ]:
results = [train_fresh(seed=None) for _ in range(3)]
for i, acc in enumerate(results, 1):
    print(f"run {i}: val_accuracy = {acc:.6f}")
print(f"spread: {max(results) - min(results):.6f}")

## 5 · The same cell, with a seed

The only change is `seed=42`: the split, the initial weights and the dropout masks are now inputs, not accidents.

In [ ]:
results = [train_fresh(seed=42) for _ in range(3)]
for i, acc in enumerate(results, 1):
    print(f"run {i}: val_accuracy = {acc:.6f}")
print(f"spread: {max(results) - min(results):.6f}")

## What to say

Nothing crashed. Same commit, same file names, three different numbers.
Which one goes in the report? Then the same cell with one change, a seed, and the three numbers agree.

**For the slides:** two of the unseeded numbers from section 4 go into slide 3.
Sections 4 and 5 are the before and after clips for demo 02 (slide 25).